In [0]:
# Package load and set up variables
import pyspark.sql.functions as f
from pyspark.sql.types import IntegerType, FloatType, StringType
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql.functions import current_timestamp
from pyspark.sql import DataFrame
from pyspark.sql.functions import regexp_replace
import datetime as dt
from datetime import timedelta, datetime, date
import pandas as pd
spark.conf.set("spark.sql.session.timeZone", "America/New_York")

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File <command-1682401951010131>, line 11
      9 import datetime as dt
     10 from datetime import timedelta, datetime, date
---> 11 from src.resources import config as c
     12 from src.resources import dates as d
     13 from src.common import email_helper

File /databricks/python_shell/dbruntime/autoreload/discoverability/hook.py:72, in AutoreloadDiscoverabilityHook.pre_run_cell.<locals>.patched_import(name, *args, **kwargs)
     66 if not self._should_hint and (
     67     (module := sys.modules.get(absolute_name)) is not None and
     68     (fname := get_allowed_file_name_or_none(module)) is not None and
     69     (mtime := os.stat(fname).st_mtime) > self.last_mtime_by_modname.get(
     70         absolute_name, float("inf")) and not self._should_hint):
     71     self._should_hint = True
---> 72 module = self._original_buil

In [0]:
acquisition_hhs = spark.read.parquet(f'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/r324398/KPF_Acquisition_HHs/acquisition_latest')
# acquisition_hhs = spark.read.option('header', 'true').csv(f'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/lookalike/acquisition_lookalike__holiday_2026_HHS')

In [0]:
final_output = acquisition_hhs.select("EHHN").distinct()

In [0]:
display(final_output.agg(f.countDistinct("EHHN")))

count(DISTINCT EHHN)
4999999


In [0]:
# class OracleHandler:

#     def __init__(self, oracle_host: str = "hamex7-scan.int.8451.com", oracle_port: str = "1521", oracle_service: str = "exa_uskrgprdh.dunnhumby.co.uk", oracle_user: str = None, oracle_password: str = None):
#         """
#         Initialize the OracleHandler with connection details.

#         Parameters:
#         oracle_host (str): The hostname of the Oracle Exadata server.
#         oracle_port (str): The port number of the Oracle Exadata server.
#         oracle_service (str): The service name of the Oracle Exadata server.
#         oracle_user (str): The Oracle user.
#         oracle_password (str): The password for the Oracle user.
#         """
#         self.oracle_host = oracle_host
#         self.oracle_port = oracle_port
#         self.oracle_service = oracle_service
#         self.oracle_user = oracle_user
#         self.oracle_password = oracle_password
#         self.jdbc_connection_string = f"jdbc:oracle:thin:@//{self.oracle_host}:{self.oracle_port}/{self.oracle_service}"

#     def read_data(self, query: str) -> DataFrame:
#         """
#         Read data from Oracle Exadata.

#         Parameters:
#         query (str): The SQL query to execute.

#         Returns:
#         DataFrame: A Spark DataFrame containing the result of the query.
#         """
#         df = spark.read.format("jdbc") \
#             .option("url", self.jdbc_connection_string) \
#             .option("query", query) \
#             .option("user", self.oracle_user) \
#             .option("password", self.oracle_password) \
#             .option("driver", "oracle.jdbc.driver.OracleDriver") \
#             .load()

#         return df

#     def read_partitioned_data(self, query: str, partition_column: str, lower_bound: int, upper_bound: int, num_partitions: int) -> DataFrame:
#         """
#         Read partitioned data from Oracle Exadata.

#         Parameters:
#         query (str): The SQL query to execute.
#         partition_column (str): The column to partition on.
#         lower_bound (int): The lower bound of the partition column.
#         upper_bound (int): The upper bound of the partition column.
#         num_partitions (int): The number of partitions.

#         Returns:
#         DataFrame: A Spark DataFrame containing the result of the query.
#         """
#         df = spark.read.format("jdbc") \
#             .option("url", self.jdbc_connection_string) \
#             .option("dbtable", f"({query})") \
#             .option("user", self.oracle_user) \
#             .option("password", self.oracle_password) \
#             .option("driver", "oracle.jdbc.driver.OracleDriver") \
#             .option("partitionColumn", partition_column) \
#             .option("lowerBound", lower_bound) \
#             .option("upperBound", upper_bound) \
#             .option("numPartitions", num_partitions) \
#             .load()

#         return df

#     def write_data(self, insert_df: DataFrame, table_name: str, mode: str = "append"):
#         """
#         Write data to an Oracle table.

#         Parameters:
#         insert_df (DataFrame): The DataFrame to write to the Oracle table.
#         table_name (str): The name of the table to write to.
#         mode (str): The mode of writing (default is "append").
#         """
#         insert_df.write.mode(mode) \
#             .format("jdbc") \
#             .option("url", self.jdbc_connection_string) \
#             .option("user", self.oracle_user) \
#             .option("password", self.oracle_password) \
#             .option("driver", "oracle.jdbc.driver.OracleDriver") \
#             .option("dbtable", table_name) \
#             .save()

# # Initialize OracleHandler
# env = "prd"
# oracle_handler = OracleHandler(oracle_user="svc_ptce_8451[BULLSEYE]", oracle_password=dbutils.secrets.get(scope=f"kv-8451-tm-dart-{env}", key="exadata-pwd"))

In [0]:
class OracleHandler:

    def __init__(self, oracle_host: str = "eusx11-scan.int.8451.com", oracle_port: str = "1521", oracle_service: str = "exa_uskrgprdh.dunnhumby.co.uk", oracle_user: str = None, oracle_password: str = None):
        """
        Initialize the OracleHandler with connection details.

        Parameters:
        oracle_host (str): The hostname of the Oracle Exadata server.
        oracle_port (str): The port number of the Oracle Exadata server.
        oracle_service (str): The service name of the Oracle Exadata server.
        oracle_user (str): The Oracle user.
        oracle_password (str): The password for the Oracle user.
        """
        self.oracle_host = oracle_host
        self.oracle_port = oracle_port
        self.oracle_service = oracle_service
        self.oracle_user = oracle_user
        self.oracle_password = oracle_password
        self.jdbc_connection_string = f"jdbc:oracle:thin:@//{self.oracle_host}:{self.oracle_port}/{self.oracle_service}"

    def read_data(self, query: str) -> DataFrame:
        """
        Read data from Oracle Exadata.

        Parameters:
        query (str): The SQL query to execute.

        Returns:
        DataFrame: A Spark DataFrame containing the result of the query.
        """
        df = spark.read.format("jdbc") \
            .option("url", self.jdbc_connection_string) \
            .option("query", query) \
            .option("user", self.oracle_user) \
            .option("password", self.oracle_password) \
            .option("driver", "oracle.jdbc.driver.OracleDriver") \
            .load()

        return df

    def read_partitioned_data(self, query: str, partition_column: str, lower_bound: int, upper_bound: int, num_partitions: int) -> DataFrame:
        """
        Read partitioned data from Oracle Exadata.

        Parameters:
        query (str): The SQL query to execute.
        partition_column (str): The column to partition on.
        lower_bound (int): The lower bound of the partition column.
        upper_bound (int): The upper bound of the partition column.
        num_partitions (int): The number of partitions.

        Returns:
        DataFrame: A Spark DataFrame containing the result of the query.
        """
        df = spark.read.format("jdbc") \
            .option("url", self.jdbc_connection_string) \
            .option("dbtable", f"({query})") \
            .option("user", self.oracle_user) \
            .option("password", self.oracle_password) \
            .option("driver", "oracle.jdbc.driver.OracleDriver") \
            .option("partitionColumn", partition_column) \
            .option("lowerBound", lower_bound) \
            .option("upperBound", upper_bound) \
            .option("numPartitions", num_partitions) \
            .load()

        return df

    def write_data(self, insert_df: DataFrame, table_name: str, mode: str = "append"):
        """
        Write data to an Oracle table.

        Parameters:
        insert_df (DataFrame): The DataFrame to write to the Oracle table.
        table_name (str): The name of the table to write to.
        mode (str): The mode of writing (default is "append").
        """
        insert_df.write.mode(mode) \
            .format("jdbc") \
            .option("url", self.jdbc_connection_string) \
            .option("user", self.oracle_user) \
            .option("password", self.oracle_password) \
            .option("driver", "oracle.jdbc.driver.OracleDriver") \
            .option("dbtable", table_name) \
            .save()

# Initialize OracleHandler
env = "prd"
oracle_handler = OracleHandler(
    oracle_host="eusx11-scan.int.8451.com",
    oracle_user="svc_ptce_8451[BULLSEYE]",
    oracle_password=dbutils.secrets.get(scope=f"kv-8451-tm-dart-{env}", key="exadata-pwd"),
)

In [0]:
oracle_handler.write_data(insert_df=final_output, table_name="BULLSEYE.KPF_ACQUISITION_LATEST", mode="overwrite")
# oracle_handler.write_data(insert_df=final_output, table_name="BULLSEYE.KPF_HOLIDAY_ACQ", mode="overwrite")

---------------------------------------------------------------------------
Py4JJavaError                             Traceback (most recent call last)
File /databricks/spark/python/pyspark/errors/exceptions/captured.py:188, in capture_sql_exception.<locals>.deco(*a, **kw)
    187 try:
--> 188     return f(*a, **kw)
    189 except Py4JJavaError as e:

File /databricks/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/protocol.py:326, in get_return_value(answer, gateway_client, target_id, name)
    325 if answer[1] == REFERENCE_TYPE:
--> 326     raise Py4JJavaError(
    327         "An error occurred while calling {0}{1}{2}.\n".
    328         format(target_id, ".", name), value)
    329 else:

Py4JJavaError: An error occurred while calling o562.save.
: java.lang.ClassNotFoundException: oracle.jdbc.driver.OracleDriver
	at java.net.URLClassLoader.findClass(URLClassLoader.java:387)
	at java.lang.ClassLoader.loadClass(ClassLoader.java:419)
	at java.lang.ClassLoader.loadClass(ClassLoader.java:35